# Inventory Decision Simulator
[Open in Colab](https://colab.research.google.com/github/ericmavigo/retail-demand-forecasting/blob/main/notebooks/04_inventory_decisions.ipynb)

Translate forecast output into order quantities, safety stock, reorder points, excess units and stockout units.

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
decisions=pd.read_parquet(ROOT/'data/processed/inventory_decisions.parquet')
summary=pd.read_csv(ROOT/'data/processed/inventory_summary.csv')

## Assumptions
- 28-day order quantity equals forecast demand.
- Safety stock targets a 95% service level.
- Lead time is seven days.
- Stockout and excess units are evaluated against actual holdout demand.
- This is a decision simulation because M5 does not contain inventory-on-hand or purchase orders.

In [ ]:
summary
px.bar(summary.melt('method',var_name='outcome',value_name='units'),x='method',y='units',color='outcome',barmode='group',title='Inventory trade-off').show()

In [ ]:
reduction=1-summary.iloc[1].stockout_units/summary.iloc[0].stockout_units
print(f'Estimated stockout-unit reduction: {reduction:.1%}')
decisions[['id','reorder_point','safety_stock','model_stockout_units','model_excess_units']].sort_values('model_stockout_units',ascending=False).head(20)

## Executive interpretation
The hybrid forecast reduces missing units while increasing excess slightly. A business user can add unit holding cost and lost-sale cost to select the preferred service level rather than optimizing forecast accuracy alone.